# Quan les dades no venen netes

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Els cinc quaderns anteriors funcionen perquè Iris és un dataset de joguina:
sense valors que falten, tot numèric, tot net. `fit()` sempre funciona a la
primera. A la feina no tens tanta sort: les dades arriben amb valors que
falten, columnes de text, escales incoherents i files duplicades. I `fit()`
peta directament, o —pitjor— no peta i et dona un resultat que no vol dir
res.

Aquest quadern fa servir el dataset `load_wine` de scikit-learn (178 vins,
13 mesures químiques, 3 varietats), fet malbé **a posta i de manera
reproduïble**, i desat com un fitxer CSV a `dades/vins_bruts.csv`. Vosaltres
el carregareu amb `pd.read_csv`, exactament com carregaríeu un fitxer que us
ha passat un company de feina.

## Fabricació del CSV (només per al docent)

Aquesta cel·la fabrica `dades/vins_bruts.csv` a partir del `load_wine` net.
A l'edició d'exercicis els alumnes **no** la tenen: només carreguen el
fitxer resultant, com si els l'hagués passat un company de feina.

In [ ]:
import os
import numpy as np
import pandas as pd

def fabrica_vins_bruts():
    """Recrea el CSV brut a partir del Wine net de scikit-learn.

    Es determinista (np.random.seed(42)): sempre fa malbe exactament les
    mateixes files i columnes.
    """
    from sklearn.datasets import load_wine

    np.random.seed(42)

    wine = load_wine(as_frame=True)
    df = wine.frame.copy().rename(columns={"target": "classe"})
    n = len(df)

    # 1) valors que falten, en tres columnes i amb percentatges diferents
    columnes_amb_nuls = {"alcohol": 0.10, "magnesium": 0.20, "proline": 0.15}
    for columna, percentatge in columnes_amb_nuls.items():
        n_nuls = int(round(n * percentatge))
        idx_nuls = np.random.choice(df.index, size=n_nuls, replace=False)
        df.loc[idx_nuls, columna] = np.nan

    # 2) una columna de text, assignada a l'atzar
    df["origen"] = np.random.choice(["Catalunya", "Rioja", "Douro"], size=n)

    # 3) una columna amb una escala absurda (com si l'haguessin mesurat en una altra unitat)
    df["nonflavanoid_phenols"] = df["nonflavanoid_phenols"] * 100000

    # 4) files duplicades
    idx_duplicar = np.random.choice(df.index, size=20, replace=False)
    df = pd.concat([df, df.loc[idx_duplicar]], ignore_index=True)

    return df


vins = fabrica_vins_bruts()
os.makedirs("dades", exist_ok=True)
vins.to_csv("dades/vins_bruts.csv", index=False)

print(f"Desat a dades/vins_bruts.csv -- files: {len(vins)}, columnes: {len(vins.columns)}")
vins.head()

A partir d'aquí, el quadern carrega el CSV exactament com ho fan els alumnes.

In [ ]:
import numpy as np
import pandas as pd

vins = pd.read_csv("dades/vins_bruts.csv")
print(f"Files: {len(vins)}, columnes: {len(vins.columns)}")
vins.head()

## Exercici 1 — El diagnòstic

Abans de tocar res, fes l'inventari dels desperfectes. Cap dataset ve amb una
"llista de defectes" enganxada: l'has de trobar tu. Amb `vins`, respon:

1. Quantes files i columnes té.
2. Quants valors nuls hi ha per columna (`.isna().sum()`), i quines columnes
   en tenen.
3. Quines columnes **no** són numèriques (`.dtypes`).
4. Quantes files duplicades hi ha (`.duplicated().sum()`).

Extra: fes `.describe()` de les columnes numèriques. Et sembla estranya
l'escala (mínim/màxim) d'alguna columna?

**Com saps que ho has fet bé:** pots dir, sense tornar a mirar el codi, tres
desperfectes concrets d'aquest dataset, cadascun amb un número exacte (no
"alguns nuls", sinó "39 nuls a `magnesium`").

In [ ]:
# Exercici 1: diagnostic

print("Files, columnes:", vins.shape)
print()

nuls = vins.isna().sum()
print("Nuls per columna (nomes les que en tenen):")
print(nuls[nuls > 0])
print()

no_numeriques = vins.dtypes[vins.dtypes == object]
print("Columnes no numeriques:", list(no_numeriques.index))
print()

print("Files duplicades:", vins.duplicated().sum())
print()

vins.describe()

## Exercici 2 — Intenta-ho a la bruta

Prova d'entrenar directament amb `vins` tal com està: separa `X` (totes les
columnes excepte `classe`) i `y` (`classe`), parteix en entrenament/examen
amb `train_test_split(test_size=0.3, random_state=42, stratify=y)`, i entrena
un `RandomForestClassifier(random_state=42)`.

**Petarà, i això és a posta.** Embolcalla el `.fit()` amb `try/except`, i
al bloc `except` imprimeix el tipus d'error i el seu missatge. Això forma
part de la lliçó: el missatge d'error diu exactament quin dels desperfectes
diagnosticats ha petat primer.

**Com saps que ho has fet bé:** t'ha sortit un missatge d'error imprès (no
una execució silenciosa), i el missatge assenyala una columna o un tipus de
dada concrets, no un error estrany sense relació.

In [ ]:
# Exercici 2: peta a proposit (forma part de l'exercici)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

try:
    X = vins.drop(columns=["classe"])
    y = vins["classe"]
    X_entrena, X_examen, y_entrena, y_examen = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    model = RandomForestClassifier(random_state=42)
    model.fit(X_entrena, y_entrena)
    print("No ha petat (inesperat)")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## Exercici 3 — Els duplicats

Elimina les files duplicades (`.drop_duplicates()`), reinicia l'índex
(`reset_index(drop=True)`), i desa el resultat a `vins_sd` (*sense
duplicats*). Comprova quantes files hi havia abans i quantes en queden
després.

Raona, en un comentari o un `print`, **per què són perillosos** els
duplicats: si la mateixa fila acaba repartida entre entrenament i examen,
l'examen ja no examina res que el model no hagi vist.

**Com saps que ho has fet bé:** `vins_sd.duplicated().sum()` dona `0`, i pots
explicar en una frase per què un duplicat repartit entre entrenament i examen
invalida l'examen.

In [ ]:
# Exercici 3: duplicats

files_abans = len(vins)
vins_sd = vins.drop_duplicates().reset_index(drop=True)
files_despres = len(vins_sd)

print(f"Files abans: {files_abans}")
print(f"Files despres de treure duplicats: {files_despres}")
print(f"Files eliminades: {files_abans - files_despres}")
print(f"Duplicats restants: {vins_sd.duplicated().sum()}")

# Per que son perillosos: si una fila duplicada acaba una copia a
# entrenament i l'altra a examen, el model "examina" una fila que
# literalment ja ha vist. La precisio a l'examen surt inflada i no
# reflecteix com es comportaria amb dades noves de veritat.

## Exercici 4 — Els valors que falten

Treballa només amb les columnes numèriques (deixa `origen` per a
l'exercici següent); l'etiqueta és `classe`. Compara tres estratègies:

- **a)** Elimina les files amb algun nul (`.dropna()`). Quantes files et
  queden?
- **b)** Omple els nuls amb la mitjana de cada columna
  (`SimpleImputer(strategy="mean")`).
- **c)** Omple els nuls amb la mediana (`SimpleImputer(strategy="median")`).

Per a cadascuna, entrena un `RandomForestClassifier(random_state=42)` (amb
`train_test_split(test_size=0.3, random_state=42, stratify=y)` sobre les
dades que resultin de l'estratègia) i anota files disponibles i precisió.

**Com saps que ho has fet bé:** la (a) et deixa amb clarament menys files que
la (b) i la (c); les altres dues fan servir totes les files de `vins_sd`.

In [ ]:
# Exercici 4: valors que falten (tres estrategies)
from sklearn.impute import SimpleImputer

cols_numeriques = [c for c in vins_sd.columns if c not in ("classe", "origen")]
resultats = []

# a) dropna
df_dropna = vins_sd.dropna(subset=cols_numeriques)
Xd, yd = df_dropna[cols_numeriques], df_dropna["classe"]
Xd_entrena, Xd_examen, yd_entrena, yd_examen = train_test_split(
    Xd, yd, test_size=0.3, random_state=42, stratify=yd
)
model_dropna = RandomForestClassifier(random_state=42).fit(Xd_entrena, yd_entrena)
resultats.append({
    "Estrategia": "dropna (elimina files)",
    "Files disponibles": len(Xd),
    "Precisio": model_dropna.score(Xd_examen, yd_examen),
})

# b) i c): mitjana i mediana, sobre les mateixes files (totes) i el mateix split
X_all = vins_sd[cols_numeriques]
y_all = vins_sd["classe"]
X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)

for nom, estrategia in [("mitjana", "mean"), ("mediana", "median")]:
    imputador = SimpleImputer(strategy=estrategia)
    X_entrena_imp = imputador.fit_transform(X_entrena)
    X_examen_imp = imputador.transform(X_examen)
    model = RandomForestClassifier(random_state=42).fit(X_entrena_imp, y_entrena)
    resultats.append({
        "Estrategia": f"SimpleImputer ({nom})",
        "Files disponibles": len(X_all),
        "Precisio": model.score(X_examen_imp, y_examen),
    })

comparativa = pd.DataFrame(resultats)
comparativa["Precisio"] = comparativa["Precisio"].map(lambda p: f"{p:.1%}")
comparativa

## Exercici 5 — La columna de text

Els models només entenen números; `origen` és text. Aplica `OneHotEncoder`
(o `pd.get_dummies`) sobre la columna `origen` de `vins_sd` i mira com una
sola columna es converteix en tres. Imprimeix els noms de les columnes noves
i la forma (`shape`) del resultat.

**Com saps que ho has fet bé:** `origen` es converteix en exactament 3
columnes (una per valor: "Catalunya", "Rioja", "Douro"), i cada fila té un 1
en una d'elles i 0 a la resta.

In [ ]:
# Exercici 5: la columna de text
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
origen_1hot = ohe.fit_transform(vins_sd[["origen"]])

print("Columnes noves:", list(ohe.get_feature_names_out(["origen"])))
print("Forma del resultat:", origen_1hot.shape)

pd.DataFrame(origen_1hot, columns=ohe.get_feature_names_out(["origen"])).head()

## Exercici 6 — El Pipeline

Refés tot el procés (imputar + codificar) amb `Pipeline` i
`ColumnTransformer`, en comptes de fer-ho a mà pas a pas. Per què importa?

**Si calcules la mitjana o la mediana per omplir els buits amb el conjunt
sencer abans de partir-lo en entrenament/examen, estàs filtrant informació
de l'examen a l'entrenament**: la mitjana ja "coneix" valors que a l'examen
haurien de ser secrets. Un `Pipeline` ho fa en l'ordre correcte: s'ajusta
(`fit`) **només** amb l'entrenament, i després s'aplica (`transform`) a
l'examen sense tornar-se a ajustar.

Construeix:

1. Un `ColumnTransformer` amb dues branques: `SimpleImputer(strategy="median")`
   per a les columnes numèriques, i `OneHotEncoder(handle_unknown="ignore")`
   per a `origen`.
2. Un `Pipeline`: `[("preprocessat", el_column_transformer), ("model",
   RandomForestClassifier(random_state=42))]`.
3. Parteix `vins_sd` en entrenament/examen **abans** de fer res més, entrena
   el pipeline sencer amb `pipe.fit(X_entrena, y_entrena)`, i avalua'l amb
   `.score()`.

**Com saps que ho has fet bé:** en tot l'exercici no crides `.fit()` ni
`.fit_transform()` fora del pipeline abans de partir les dades; l'únic ajust
és `pipe.fit(X_entrena, y_entrena)`.

In [ ]:
# Exercici 6: Pipeline + ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = vins_sd.drop(columns=["classe"])
y = vins_sd["classe"]

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

cols_numeriques = [c for c in X.columns if c != "origen"]

preprocessat = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), cols_numeriques),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["origen"]),
])

pipe = Pipeline([
    ("preprocessat", preprocessat),
    ("model", RandomForestClassifier(random_state=42)),
])

pipe.fit(X_entrena, y_entrena)
precisio_pipeline = pipe.score(X_examen, y_examen)

print(f"Precisio del pipeline: {precisio_pipeline:.1%}")

## Exercici 7 — El resultat final

Compara la precisió del teu pipeline (exercici 6) amb la d'un
`RandomForestClassifier(random_state=42)` entrenat sobre el `load_wine`
**net i original** de scikit-learn (sense cap desperfecte), amb el mateix
`train_test_split(test_size=0.3, random_state=42, stratify=y)`.

Han de quedar a prop l'una de l'altra. Si la diferència és de desenes de
punts, alguna cosa s'ha perdut pel camí (una estratègia mal triada, una fuita
d'informació, un error).

**Com saps que ho has fet bé:** la diferència entre les dues precisions es
mesura en pocs punts percentuals, no en desenes.

In [ ]:
# Exercici 7: el resultat final
from sklearn.datasets import load_wine

wine_net = load_wine(as_frame=True)
Xc = wine_net.frame.drop(columns=["target"])
yc = wine_net.frame["target"]

Xc_entrena, Xc_examen, yc_entrena, yc_examen = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc
)

model_net = RandomForestClassifier(random_state=42).fit(Xc_entrena, yc_entrena)
precisio_net = model_net.score(Xc_examen, yc_examen)

print(f"Precisio amb Wine net original: {precisio_net:.1%}")
print(f"Precisio amb el pipeline sobre dades brutes: {precisio_pipeline:.1%}")
print(f"Diferencia: {abs(precisio_net - precisio_pipeline):.1%}")

## Resum

- Les dades brutes trenquen `fit()` de dues maneres: **petant** (columnes de
  text, valors `NaN`) o, més perillós, **no petant** i donant un resultat que
  no vol dir res.
- El primer pas sempre és **diagnosticar**: files, columnes, nuls per
  columna, tipus de dada, duplicats. No es pot arreglar el que no s'ha
  mirat.
- Els **duplicats** s'han d'eliminar abans de partir en entrenament/examen:
  si no, la mateixa fila pot acabar als dos costats i l'examen deixa de
  ser-ho.
- No hi ha una única manera correcta d'omplir **valors que falten**:
  eliminar files hi perd informació, la mitjana i la mediana en conserven
  més però n'inventen una part. Cal comparar-les.
- Els models només entenen números: una columna de **text** s'ha de
  codificar (`OneHotEncoder` o `pd.get_dummies`) abans d'entrenar.
- **`Pipeline` + `ColumnTransformer`** no és només comoditat: és l'única
  manera de garantir que la imputació i la codificació s'ajusten amb
  l'entrenament i s'apliquen (sense reajustar-se) a l'examen. Fer-ho a mà en
  l'ordre equivocat filtra informació de l'examen cap a l'entrenament, i el
  resultat sembla millor del que realment és.